In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import classification_report, f1_score, accuracy_score
import time, json, warnings, os
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz   : {device}")


Cihaz   : cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
LABEL_NAMES = {
    0: "Yardım Talebi",
    1: "Kayıp Bildirimi",
    2: "Altyapı Hasarı",
    3: "Bağış/Koordinasyon",
    4: "Diğer/İlgisiz"
}

train_df = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/processed/train.csv')
val_df   = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/processed/val.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/bdm_proje/data/processed/test.csv')

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")
print(f"\nTrain etiket dağılımı:")
print(train_df['label_id'].value_counts().sort_index())

Train : 633
Val   : 80
Test  : 80

Train etiket dağılımı:
label_id
0    231
1     78
2     80
3     83
4    161
Name: count, dtype: int64


In [4]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ============================================================
# 4-BIT QUANTIZATION AYARI
# ============================================================
# nf4 → "Normal Float 4" — QLoRA makalesi tarafından önerilen format
# double_quant → quantization parametrelerini de quantize eder, ekstra bellek tasarrufu
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True
)

print("Model yükleniyor (4-bit QLoRA)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"  # eğitimde sağdan padding

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto"
)

# Gradient checkpoint — VRAM daha da düşer, biraz yavaşlar ama güvenli
model.gradient_checkpointing_enable()
model.config.use_cache = False  # gradient checkpoint ile uyumsuz, kapatıyoruz

print(f"✅ Model yüklendi!")
print(f"VRAM kullanımı: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Model yükleniyor (4-bit QLoRA)...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model yüklendi!
VRAM kullanımı: 0.00 GB


In [5]:

# ============================================================
# LORA AYARI
# ============================================================
# r (rank): adaptör matrisinin boyutu. Büyük = daha fazla parametre = daha iyi ama yavaş
# alpha: scaling faktörü. Genellikle r*2 yapılır
# target_modules: hangi katmanlara adaptör ekleneceği
# dropout: overfitting önleme

lora_config = LoraConfig(
    r                = 16,
    lora_alpha       = 32,
    target_modules   = ["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout     = 0.05,
    bias             = "none",
    task_type        = TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# Kaç parametre eğitiyoruz?
toplam     = sum(p.numel() for p in model.parameters())
egitilen   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Toplam parametre    : {toplam/1e6:.1f}M")
print(f"Eğitilen parametre  : {egitilen/1e6:.2f}M")
print(f"Eğitim oranı        : %{egitilen/toplam*100:.2f}")

Toplam parametre    : 620.1M
Eğitilen parametre  : 4.51M
Eğitim oranı        : %0.73


In [6]:
def format_prompt(row):
    """
    Her tweet'i instruction-following formatına çevirir.
    Model bu formatta hem soruyu hem cevabı görür — böyle öğrenir.
    """
    sistem = (
        "Sen bir deprem acil durum sınıflandırma asistanısın. "
        "Verilen tweeti aşağıdaki kategorilerden birine koy. "
        "SADECE kategori adını yaz, başka hiçbir şey yazma."
    )

    kullanici = (
        f"Tweet: {row['tweet']}\n\n"
        "Kategoriler:\n"
        "- Yardım Talebi\n"
        "- Kayıp Bildirimi\n"
        "- Altyapı Hasarı\n"
        "- Bağış/Koordinasyon\n"
        "- Diğer/İlgisiz\n\n"
        "Kategori:"
    )

    asistan = LABEL_NAMES[row['label_id']]

    # TinyLlama chat formatı
    return (
        f"<|system|>\n{sistem}</s>\n"
        f"<|user|>\n{kullanici}</s>\n"
        f"<|assistant|>\n{asistan}</s>"
    )


# Tüm setlere uygula
train_df['prompt'] = train_df.apply(format_prompt, axis=1)
val_df['prompt']   = val_df.apply(format_prompt, axis=1)

# Örnek göster
print("=== ÖRNEK PROMPT ===")
print(train_df['prompt'].iloc[0])
print(f"\nOrtalama token uzunluğu: {train_df['prompt'].apply(lambda x: len(x.split())).mean():.0f} kelime")

=== ÖRNEK PROMPT ===
<|system|>
Sen bir deprem acil durum sınıflandırma asistanısın. Verilen tweeti aşağıdaki kategorilerden birine koy. SADECE kategori adını yaz, başka hiçbir şey yazma.</s>
<|user|>
Tweet: Beyoğlu mahallesi, şehit İsmail Orçan bulvarı no:32 Türkoğlu / Kahramanmaraş +905388623929  Tanju Ulu  Acil çadır desteğine ihtiyaç vardır. Yardımcı olun lütfen. Belirtilen adrese yardım gitmemiş. Güncel bilgidir. İletişim halindeyiz.

Kategoriler:
- Yardım Talebi
- Kayıp Bildirimi
- Altyapı Hasarı
- Bağış/Koordinasyon
- Diğer/İlgisiz

Kategori:</s>
<|assistant|>
Yardım Talebi</s>

Ortalama token uzunluğu: 63 kelime


In [7]:
from datasets import Dataset

# trl 1.4.0 → 'prompt' + 'completion' formatı bekliyor
def format_prompt_completion(row):
    sistem = (
        "Sen bir deprem acil durum sınıflandırma asistanısın. "
        "Verilen tweeti aşağıdaki kategorilerden birine koy. "
        "SADECE kategori adını yaz, başka hiçbir şey yazma."
    )
    kullanici = (
        f"Tweet: {row['tweet']}\n\n"
        "Kategoriler:\n"
        "- Yardım Talebi\n"
        "- Kayıp Bildirimi\n"
        "- Altyapı Hasarı\n"
        "- Bağış/Koordinasyon\n"
        "- Diğer/İlgisiz\n\n"
        "Kategori:"
    )
    prompt = (
        f"<|system|>\n{sistem}</s>\n"
        f"<|user|>\n{kullanici}</s>\n"
        f"<|assistant|>\n"
    )
    completion = f"{LABEL_NAMES[row['label_id']]}</s>"

    return {"prompt": prompt, "completion": completion}

train_rows = [format_prompt_completion(row) for _, row in train_df.iterrows()]
val_rows   = [format_prompt_completion(row) for _, row in val_df.iterrows()]

train_dataset = Dataset.from_list(train_rows)
val_dataset   = Dataset.from_list(val_rows)

print(f"Train: {len(train_dataset)}")
print(f"Val  : {len(val_dataset)}")
print(f"\nÖrnek:")
print(f"PROMPT     : {train_dataset[0]['prompt'][:100]}...")
print(f"COMPLETION : {train_dataset[0]['completion']}")

Train: 633
Val  : 80

Örnek:
PROMPT     : <|system|>
Sen bir deprem acil durum sınıflandırma asistanısın. Verilen tweeti aşağıdaki kategoriler...
COMPLETION : Yardım Talebi</s>


In [9]:
OUTPUT_DIR = "/content/drive/MyDrive/bdm_proje/models/tinyllama_qlora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

sft_config = SFTConfig(
    output_dir            = OUTPUT_DIR,

    num_train_epochs      = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,

    learning_rate         = 2e-4,
    weight_decay          = 0.01,
    warmup_ratio          = 0.03,
    lr_scheduler_type     = "cosine",

    eval_strategy         = "steps",
    eval_steps            = 50,
    save_strategy         = "steps",
    save_steps            = 50,
    load_best_model_at_end= True,
    metric_for_best_model = "eval_loss",

    logging_steps         = 10,
    report_to             = "none",

    # trl 1.4.0 — prompt/completion formatı
    dataset_text_field    = "prompt",
    completion_only_loss  = True,   # sadece completion kısmının loss'u hesaplanır

    seed                  = SEED,
    fp16                  = False,
    bf16                  = True,
)

trainer = SFTTrainer(
    model           = model,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    args            = sft_config,
    processing_class= tokenizer,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Eğitim başlıyor...")
print(f"  Epoch           : {sft_config.num_train_epochs}")
print(f"  Batch size      : {sft_config.per_device_train_batch_size}")
print(f"  Gradient accum  : {sft_config.gradient_accumulation_steps}")
print(f"  Effective batch : {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Learning rate   : {sft_config.learning_rate}")
print()

train_result = trainer.train()

print(f"\n✅ Eğitim tamamlandı!")
print(f"   Toplam adım    : {train_result.global_step}")
print(f"   Train loss     : {train_result.training_loss:.4f}")
print(f"   Eğitim süresi  : {train_result.metrics['train_runtime']:.0f} sn")

ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU

In [ ]:
import trl
print(trl.__version__)

1.4.0
